# Session 6 — GPU: JAX/Flax Throughput Benchmark

## Background

Sessions 1–5 reached the GPU and TPU through PyTorch and PyTorch/XLA. JAX takes a different approach: functional programming model, explicit parameter trees, and `jax.jit` compiling the full train step (forward + backward + optimizer update) into a single XLA program. On GPU, JAX uses the CUDA XLA backend — the same XLA compiler stack, but targeting cuBLAS/cuDNN rather than TPU MXUs.

Session 1 measured PyTorch-native GPU throughput as the baseline. This notebook measures JAX/Flax throughput on the same hardware using the equivalent architecture (`FlaxTransformerBlock` in `flax_model.py`). Session 6's analysis will overlay the two curves to answer whether the framework choice is a performance decision or an ergonomics decision on GPU.

## Goal

Measure JAX/Flax training throughput on the NVIDIA L4 across the same batch sweep as Session 1. Record first-step compilation time per batch size. Produce `results/gpu_jax.json` for comparison in `03-analysis.ipynb`.

## Question

How does JAX/Flax GPU throughput compare to PyTorch native (Session 1), and what is the per-shape XLA compilation cost?

---

**Runtime:** GPU (NVIDIA L4 or equivalent with CUDA 12)

After running, results are saved to `results/gpu_jax.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
# Install JAX with CUDA 12 support, Flax, and Optax.
# Skip if already installed in this runtime.
import importlib, subprocess, sys

def _need(pkg):
    try:
        importlib.import_module(pkg)
        return False
    except ImportError:
        return True

if _need("jax"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "jax[cuda12]", "flax", "optax",
    ])
    print("Installed jax[cuda12], flax, optax")
else:
    print("JAX already available — skipping install")

JAX already available — skipping install


In [ ]:
import jax
import jax.numpy as jnp
import flax
import optax
import time

gpu_devices = jax.devices("gpu")
assert len(gpu_devices) > 0, "No GPU found. Ensure CUDA 12 drivers are installed and jax[cuda12] is used."

device = gpu_devices[0]
print(f"JAX version  : {jax.__version__}")
print(f"Flax version : {flax.__version__}")
print(f"Optax version: {optax.__version__}")
print(f"GPU device   : {device}")
print(f"All devices  : {jax.devices()}")

JAX version  : 0.9.0.1
Flax version : 0.12.4
Optax version: 0.2.6
GPU device   : cuda:0
All devices  : [CudaDevice(id=0)]


In [ ]:
import sys; sys.path.insert(0, ".")
from flax_model import FlaxTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD, SEQ_LEN

# ── Session 6 config ──────────────────────────────────────────────────────────
BATCH_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]   # mirror Session 1
WARMUP      = 5

# Large batches have long backward passes — use fewer steps to avoid hangs.
STEPS_FOR_BATCH = {4: 50, 8: 50, 16: 50, 32: 50, 64: 50,
                   128: 50, 256: 50, 512: 20, 1024: 20}

model     = FlaxTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
optimizer = optax.adam(1e-4)

# ── JIT-compiled train step ───────────────────────────────────────────────────
# jax.jit traces this function once per unique input shape and caches the
# compiled XLA program. The first call for a new (batch_size, seq_len) shape
# incurs the compilation cost; subsequent calls reuse the cached program.
@jax.jit
def train_step(params, opt_state, x, y):
    def loss_fn(p):
        pred = model.apply(p, x)
        return jnp.mean((pred - y) ** 2)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

print(f"Model        : FlaxTransformerBlock  d_model={D_MODEL}  nhead={N_HEAD}  ff={DIM_FEEDFORWARD}")
print(f"Seq len      : {SEQ_LEN}")
print(f"Batch sizes  : {BATCH_SIZES}")
print(f"Steps        : {STEPS_FOR_BATCH}")
print("train_step defined (first call per batch size will include XLA compilation).")

Model        : FlaxTransformerBlock  d_model=512  nhead=8  ff=2048
Seq len      : 128
Batch sizes  : [4, 8, 16, 32, 64, 128, 256, 512, 1024]
Steps        : {4: 50, 8: 50, 16: 50, 32: 50, 64: 50, 128: 50, 256: 50, 512: 20, 1024: 20}
train_step defined (first call per batch size will include XLA compilation).


In [ ]:
def benchmark(batch_size, n_steps, warmup):
    """Benchmark one batch size.  Returns (compile_time_s, throughput, latency_ms)."""
    init_key = jax.random.PRNGKey(42)
    dummy_x  = jnp.ones((batch_size, SEQ_LEN, D_MODEL))
    params    = model.init(init_key, dummy_x)
    opt_state = optimizer.init(params)

    rng = jax.random.PRNGKey(0)

    # ── Step 0: first call = XLA compilation + execution ─────────────────────
    rng, k1, k2 = jax.random.split(rng, 3)
    x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
    y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))

    t0 = time.perf_counter()
    params, opt_state, loss = train_step(params, opt_state, x, y)
    jax.block_until_ready(loss)
    compile_time = time.perf_counter() - t0

    # ── Warmup: remaining warmup-1 steps ──────────────────────────────────────
    for _ in range(warmup - 1):
        rng, k1, k2 = jax.random.split(rng, 3)
        x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
        y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))
        params, opt_state, loss = train_step(params, opt_state, x, y)
        jax.block_until_ready(loss)

    # ── Timed steps ───────────────────────────────────────────────────────────
    elapsed = 0.0
    for _ in range(n_steps):
        rng, k1, k2 = jax.random.split(rng, 3)
        x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
        y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))
        t0 = time.perf_counter()
        params, opt_state, loss = train_step(params, opt_state, x, y)
        jax.block_until_ready(loss)
        elapsed += time.perf_counter() - t0

    throughput = (n_steps * batch_size) / elapsed
    latency_ms = (elapsed / n_steps) * 1000
    return round(compile_time, 3), round(throughput, 1), round(latency_ms, 2)

print("Benchmark function defined.")

Benchmark function defined.


In [ ]:
results       = {}
compile_times = {}

print(f"{'batch':<8}  {'steps':>6}  {'compile':>10}  {'throughput':>15}  {'latency':>10}")
print(f"{'─'*8}  {'─'*6}  {'─'*10}  {'─'*15}  {'─'*10}")

for bs in BATCH_SIZES:
    n = STEPS_FOR_BATCH[bs]
    try:
        ctime, tput, lat = benchmark(bs, n, WARMUP)
        results[str(bs)]       = {"throughput": tput, "latency_ms": lat}
        compile_times[str(bs)] = ctime
        print(f"{bs:<8}  {n:>6}  {ctime:>9.2f}s  {tput:>14,.0f}  {lat:>9.1f} ms")
    except Exception as e:
        results[str(bs)]       = None
        compile_times[str(bs)] = None
        print(f"{bs:<8}  {n:>6}  {'ERROR':>10}  {str(e)[:40]}")

batch      steps     compile       throughput     latency
────────  ──────  ──────────  ───────────────  ──────────
4             50      15.63s           3,186        1.3 ms
8             50      18.02s           4,409        1.8 ms
16            50      19.52s           5,946        2.7 ms
32            50      22.55s           6,070        5.3 ms
64            50      18.81s           5,304       12.1 ms
128           50      19.55s           4,823       26.5 ms
256           50      20.43s           4,660       54.9 ms
512           20      22.04s           4,571      112.0 ms
1024          20      26.53s           4,463      229.4 ms


In [ ]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":        "GPU",
    "framework":       "JAX/Flax",
    "device_name":     str(jax.devices("gpu")[0]),
    "jax_version":     jax.__version__,
    "flax_version":    flax.__version__,
    "session":         "6",
    "batch_sizes":     BATCH_SIZES,
    "seq_len":         SEQ_LEN,
    "timestamp":       datetime.now(timezone.utc).isoformat(),
    "compile_times_s": compile_times,
    "results":         results,
}
path = "results/gpu_jax.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")

Saved → results/gpu_jax.json
